In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Ejercicio 1

**Modelo de análisis factorial** con $k$ factores:

$$
\mathbf x_i=\boldsymbol\mu+\Lambda\mathbf z_i+\boldsymbol\varepsilon_i,\qquad \mathbf z_i\sim\mathcal N(\mathbf 0,I_k),\qquad \boldsymbol\varepsilon_i\sim\mathcal N(\mathbf 0,\Psi),
$$

con $\Lambda\in\mathbb R^{p\times k}$ (cargas) y $\Psi=\operatorname{diag}(\psi_1,\dots,\psi_p)$ (varianzas específicas). Marginalmente, $\mathbf x\sim\mathcal N(\boldsymbol\mu,\ C)$ con $C=\Lambda\Lambda^\top+\Psi$.

El MLE de $\boldsymbol\mu$ es la media muestral, así que trabajamos con los datos centrados $\tilde{\mathbf x}_i=\mathbf x_i-\bar{\mathbf x}$ y su covarianza $S=\frac1n\sum_i\tilde{\mathbf x}_i\tilde{\mathbf x}_i^\top$. La log-verosimilitud es

$$
\ell(\Lambda,\Psi)=-\frac n2\Big[p\log(2\pi)+\log|C|+\operatorname{tr}(C^{-1}S)\Big].
$$

**Algoritmo EM.** Las variables latentes son los factores $\mathbf z_i$. A partir de $\Lambda^{(0)}$, $\Psi^{(0)}$, repetir:

1. **Paso E**: la distribución de $\mathbf z_i$ dado $\mathbf x_i$ es normal, con

$$
G=\big(I_k+\Lambda^\top\Psi^{-1}\Lambda\big)^{-1},\qquad
\mathbb E[\mathbf z_i\mid\mathbf x_i]=G\Lambda^\top\Psi^{-1}\tilde{\mathbf x}_i,\qquad
\mathbb E[\mathbf z_i\mathbf z_i^\top\mid\mathbf x_i]=G+\mathbb E[\mathbf z_i\mid\mathbf x_i]\,\mathbb E[\mathbf z_i\mid\mathbf x_i]^\top .
$$

   ($G$ es la covarianza de $\mathbf z_i$ dado $\mathbf x_i$, la misma para todas las observaciones, de forma `(k, k)`: invertirla es mucho más barato que invertir la matriz $C$ de $p\times p$.)

2. **Paso M**: con $E_z$ la matriz `(n, k)` cuyas filas son los $\mathbb E[\mathbf z_i\mid\mathbf x_i]^\top$ y $\tilde X$ la matriz `(n, p)` de datos centrados,

$$
\Lambda=\big(\tilde X^\top E_z\big)\Big(\textstyle\sum_i\mathbb E[\mathbf z_i\mathbf z_i^\top\mid\mathbf x_i]\Big)^{-1}=\big(\tilde X^\top E_z\big)\big(nG+E_z^\top E_z\big)^{-1},
\qquad
\Psi=\operatorname{diag}\Big(S-\tfrac1n\Lambda E_z^\top\tilde X\Big).
$$

**Inicialización**: $\Psi^{(0)}=\operatorname{diag}(S)$ y $\Lambda^{(0)}$ con entradas aleatorias de la escala de los datos.

**Criterio de parada**: incremento de $\ell$ menor que `tol` (como en `myGMM`); si se llega a `max_iter`, se avisa.

El método `transform` devuelve los **factores estimados** de cada observación, $\mathbb E[\mathbf z_i\mid\mathbf x_i]$.

In [ ]:
class myFactorAnalysis:
    """
    Análisis factorial ajustado con el algoritmo EM.
    """
    def __init__(self, k, max_iter=10000, tol=1e-8, random_state=None):
        """
        Inicializa el modelo.

        Parámetros:
        - k: número de factores
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre el incremento de la log-verosimilitud
        - random_state: semilla para la inicialización de Λ
        """

        self.k = k
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

        # Resultados del ajuste
        self.mu = None  # media (p,)
        self.Lambda = None  # cargas (p, k)
        self.psi = None  # varianzas específicas, diagonal de Ψ (p,)
        self.ell = None  # log-verosimilitud final
        self.ell_hist = None  # log-verosimilitud en cada iteración
        self.n_iter_ = None  # iteraciones realizadas

    def log_verosimilitud(self, S, n, Lambda, psi):
        """
        ℓ = -n/2 [p log(2π) + log|C| + tr(C^{-1} S)], con C = Λ Λ^T + Ψ.

        Parámetros:
        - S: covarianza muestral (p, p)
        - n: cantidad de observaciones
        - Lambda: (p, k)
        - psi: (p,)

        Retorna:
        - ell: float
        """

        # TODO: C = Λ Λ^T + Ψ; log|C| con np.linalg.slogdet; tr(C^{-1} S) con np.linalg.solve

        p = S.shape[0]
        C = ...  # (p, p)
        return ...

    def paso_E(self, Xc, Lambda, psi):
        """
        Paso E: G y las medias condicionales E[z_i | x_i].

        Parámetros:
        - Xc: datos centrados (n, p)
        - Lambda: (p, k)
        - psi: (p,)

        Retorna:
        - Ez: E[z_i | x_i] en cada fila (n, k)
        - G: covarianza de z_i dado x_i (k, k)
        """

        # TODO: G = (I + Λ^T Ψ^{-1} Λ)^{-1}; E[z_i | x_i] = G Λ^T Ψ^{-1} x̃_i para todas las filas a la vez

        # Ψ^{-1} Λ sin armar Ψ: dividir cada fila de Λ por su ψ_j, (p, k) / (p, 1)
        G = ...  # (k, k)
        # (n, p) / (p,) -> (n, p); (n, p) @ (p, k) @ (k, k) -> (n, k)
        Ez = ...
        return Ez, G

    def paso_M(self, Xc, S, Ez, G):
        """
        Paso M: actualización de Λ y Ψ.

        Parámetros:
        - Xc: datos centrados (n, p)
        - S: covarianza muestral (p, p)
        - Ez: (n, k)
        - G: (k, k)

        Retorna:
        - Lambda: (p, k)
        - psi: (p,)
        """

        # TODO: Λ = (X̃^T E_z)(n G + E_z^T E_z)^{-1}; Ψ = diag(S - Λ E_z^T X̃ / n)

        n = Xc.shape[0]
        # (p, n) @ (n, k) -> (p, k); (k, k) + (k, n) @ (n, k) -> (k, k)
        Lambda = ...  # (p, k)
        # (p, k) @ (k, n) @ (n, p) -> (p, p)
        psi = ...  # (p,)
        return Lambda, psi

    def fit(self, X):
        """
        Ajusta el modelo alternando los pasos E y M.

        Parámetros:
        - X: (n, p)

        Retorna:
        - self
        """

        # TODO:
        # - μ = media de X; datos centrados y S
        # - Inicializar Ψ = diag(S) y Λ aleatoria (rng.normal escalada por el desvío de cada variable)
        # - Alternar pasos E y M, guardando ℓ; parar si el incremento es menor que tol; si se llega a max_iter, avisar

        rng = np.random.default_rng(self.random_state)
        n, p = X.shape
        self.mu = ...  # (p,)
        Xc = ...  # (n, p)
        S = ...  # (p, p)

        psi = ...  # (p,)
        # (p, k) * (p, 1) -> (p, k)
        Lambda = rng.normal(size=(p, self.k)) * np.sqrt(psi)[:, None]
        ell = self.log_verosimilitud(S, n, Lambda, psi)
        ell_hist = [ell]

        for t in range(self.max_iter):
            Ez, G = ...
            Lambda, psi = ...
            ell_nuevo = ...
            ell_hist.append(ell_nuevo)

            incremento = ...
            ell = ell_nuevo
            if ...:
                break
        else:
            warnings.warn(f"myFactorAnalysis no convergió en {self.max_iter} iteraciones")

        self.Lambda, self.psi = Lambda, psi
        self.ell = ell
        self.ell_hist = ell_hist
        self.n_iter_ = t + 1
        return self

    def transform(self, X):
        """
        Factores estimados E[z_i | x_i] de cada observación.

        Parámetros:
        - X: (m, p)

        Retorna:
        - Ez: (m, k)
        """

        # TODO

        Ez, _ = ...
        return Ez

##### Solución

In [ ]:
class myFactorAnalysis:
    """
    Análisis factorial ajustado con el algoritmo EM.
    """
    def __init__(self, k, max_iter=10000, tol=1e-8, random_state=None):
        """
        Inicializa el modelo.

        Parámetros:
        - k: número de factores
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre el incremento de la log-verosimilitud
        - random_state: semilla para la inicialización de Λ
        """

        self.k = k
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

        # Resultados del ajuste
        self.mu = None  # media (p,)
        self.Lambda = None  # cargas (p, k)
        self.psi = None  # varianzas específicas, diagonal de Ψ (p,)
        self.ell = None  # log-verosimilitud final
        self.ell_hist = None  # log-verosimilitud en cada iteración
        self.n_iter_ = None  # iteraciones realizadas

    def log_verosimilitud(self, S, n, Lambda, psi):
        """
        ℓ = -n/2 [p log(2π) + log|C| + tr(C^{-1} S)], con C = Λ Λ^T + Ψ.

        Parámetros:
        - S: covarianza muestral (p, p)
        - n: cantidad de observaciones
        - Lambda: (p, k)
        - psi: (p,)

        Retorna:
        - ell: float
        """

        p = S.shape[0]
        C = Lambda @ Lambda.T + np.diag(psi)  # (p, p)
        return -n / 2 * (p * np.log(2 * np.pi) + np.linalg.slogdet(C)[1] + np.trace(np.linalg.solve(C, S)))

    def paso_E(self, Xc, Lambda, psi):
        """
        Paso E: G y las medias condicionales E[z_i | x_i].

        Parámetros:
        - Xc: datos centrados (n, p)
        - Lambda: (p, k)
        - psi: (p,)

        Retorna:
        - Ez: E[z_i | x_i] en cada fila (n, k)
        - G: covarianza de z_i dado x_i (k, k)
        """

        # Ψ^{-1} Λ sin armar Ψ: dividir cada fila de Λ por su ψ_j, (p, k) / (p, 1)
        G = np.linalg.inv(np.eye(Lambda.shape[1]) + Lambda.T @ (Lambda / psi[:, None]))  # (k, k)
        # (n, p) / (p,) -> (n, p); (n, p) @ (p, k) @ (k, k) -> (n, k)
        Ez = (Xc / psi) @ Lambda @ G
        return Ez, G

    def paso_M(self, Xc, S, Ez, G):
        """
        Paso M: actualización de Λ y Ψ.

        Parámetros:
        - Xc: datos centrados (n, p)
        - S: covarianza muestral (p, p)
        - Ez: (n, k)
        - G: (k, k)

        Retorna:
        - Lambda: (p, k)
        - psi: (p,)
        """

        n = Xc.shape[0]
        # (p, n) @ (n, k) -> (p, k); (k, k) + (k, n) @ (n, k) -> (k, k)
        Lambda = (Xc.T @ Ez) @ np.linalg.inv(n * G + Ez.T @ Ez)  # (p, k)
        # (p, k) @ (k, n) @ (n, p) -> (p, p)
        psi = np.diag(S - Lambda @ (Ez.T @ Xc) / n).copy()  # (p,)
        return Lambda, psi

    def fit(self, X):
        """
        Ajusta el modelo alternando los pasos E y M.

        Parámetros:
        - X: (n, p)

        Retorna:
        - self
        """

        rng = np.random.default_rng(self.random_state)
        n, p = X.shape
        self.mu = np.mean(X, axis=0)  # (p,)
        Xc = X - self.mu  # (n, p)
        S = Xc.T @ Xc / n  # (p, p)

        psi = np.diag(S).copy()  # (p,)
        # (p, k) * (p, 1) -> (p, k)
        Lambda = rng.normal(size=(p, self.k)) * np.sqrt(psi)[:, None]
        ell = self.log_verosimilitud(S, n, Lambda, psi)
        ell_hist = [ell]

        for t in range(self.max_iter):
            Ez, G = self.paso_E(Xc, Lambda, psi)
            Lambda, psi = self.paso_M(Xc, S, Ez, G)
            ell_nuevo = self.log_verosimilitud(S, n, Lambda, psi)
            ell_hist.append(ell_nuevo)

            incremento = ell_nuevo - ell
            ell = ell_nuevo
            if incremento < self.tol:
                break
        else:
            warnings.warn(f"myFactorAnalysis no convergió en {self.max_iter} iteraciones")

        self.Lambda, self.psi = Lambda, psi
        self.ell = ell
        self.ell_hist = ell_hist
        self.n_iter_ = t + 1
        return self

    def transform(self, X):
        """
        Factores estimados E[z_i | x_i] de cada observación.

        Parámetros:
        - X: (m, p)

        Retorna:
        - Ez: (m, k)
        """

        Ez, _ = self.paso_E(X - self.mu, self.Lambda, self.psi)
        return Ez

##### Verificación con datos simulados

**Simulación.** Elegimos cargas con *estructura simple*: las variables 1 a 3 dependen sobre todo del factor 1 y las variables 4 a 6, del factor 2.

In [ ]:
p, k, n = 6, 2, 1000

Lambda = np.array([
    [0.9, 0.0],
    [0.8, 0.1],
    [0.7, 0.0],
    [0.0, 0.9],
    [0.1, 0.8],
    [0.0, 0.7],
])  # (p, k)
psi = np.array([0.2, 0.3, 0.4, 0.2, 0.3, 0.4])  # diagonal de Ψ (p,)
mu = np.array([1.0, 2.0, 3.0, -1.0, 0.0, 5.0])  # (p,)

def simular_fa(n, mu, Lambda, psi, rng):
    """
    Simula n observaciones x_i = μ + Λ z_i + ε_i.

    Retorna:
    - X: (n, p)
    - Z: factores (n, k)
    """
    p, k = Lambda.shape
    Z = rng.standard_normal((n, k))  # z_i ~ N(0, I)
    E = rng.standard_normal((n, p)) * np.sqrt(psi)  # ε_i ~ N(0, Ψ)
    return mu + Z @ Lambda.T + E, Z

rng = np.random.default_rng(0)
X, Z = simular_fa(n, mu, Lambda, psi, rng)
print("X:", X.shape)

In [ ]:
from sklearn.decomposition import FactorAnalysis

fa = myFactorAnalysis(k=2, random_state=0).fit(X)
fa_sk = FactorAnalysis(n_components=2, tol=1e-10, max_iter=10000).fit(X)

print(f"myFactorAnalysis: {fa.n_iter_} iteraciones, ℓ = {fa.ell:.6f}")
print(f"FactorAnalysis: ℓ = {fa_sk.score(X) * n:.6f}")
print(f"¿ℓ no decreciente? {np.all(np.diff(fa.ell_hist) >= -1e-9)}")
print(f"máx. |Λ̂Λ̂^T - Λ̂_sk Λ̂_sk^T| = {np.abs(fa.Lambda @ fa.Lambda.T - fa_sk.components_.T @ fa_sk.components_).max():.1e}")
print(f"máx. |ψ̂ - ψ̂_sk| = {np.abs(fa.psi - fa_sk.noise_variance_).max():.1e}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(fa.ell_hist)
ax.set_xlabel("iteración t")
ax.set_ylabel(r"$\ell(\theta^{(t)})$")
ax.set_ylim(fa.ell_hist[1] - 20, fa.ell + 20)
plt.show()

En `scikit-learn`, `FactorAnalysis` maximiza la misma verosimilitud con otro algoritmo (iteraciones basadas en la SVD). Sus atributos:

| atributo | qué es | forma |
|---|---|---|
| `components_` | $\Lambda^\top$ (cargas, **traspuestas**) | `(k, p)` |
| `noise_variance_` | diagonal de $\Psi$ | `(p,)` |
| `mean_` | $\boldsymbol\mu$ | `(p,)` |
| `get_covariance()` | $\Lambda\Lambda^\top+\Psi$ | `(p, p)` |

---

## Ejercicio 2

Una función para mostrar matrices de cargas como mapas de calor, con el valor de cada entrada:

In [ ]:
def graficar_cargas(matrices, titulos, vmax=1.0):
    """
    Mapas de calor de matrices de cargas (p, k), una al lado de la otra.
    """
    fig, axes = plt.subplots(1, len(matrices), figsize=(2.6 * len(matrices), 4))
    for ax, L, titulo in zip(np.atleast_1d(axes), matrices, titulos):
        ax.imshow(L, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        for (i, j), v in np.ndenumerate(L):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9)
        ax.set_xticks(range(L.shape[1]), [f"f{j + 1}" for j in range(L.shape[1])])
        ax.set_yticks(range(L.shape[0]), [f"x{i + 1}" for i in range(L.shape[0])])
        ax.set_title(titulo, fontsize=10)
    plt.tight_layout()
    return fig

#### a) Ajuste sin rotación

In [ ]:
Lambda_hat = fa.Lambda  # (p, k), de myFactorAnalysis

graficar_cargas([Lambda, Lambda_hat], [r"$\Lambda$ (simulación)", r"$\hat\Lambda$ (myFactorAnalysis)"])
plt.show()

LLt = Lambda @ Lambda.T
LLt_hat = Lambda_hat @ Lambda_hat.T
print(f"máx. |Λ̂ - Λ| = {np.max(np.abs(Lambda_hat - Lambda)):.3f}")
print(f"máx. |Λ̂Λ̂^T - ΛΛ^T| = {np.max(np.abs(LLt_hat - LLt)):.3f}")
print(f"máx. |ΛΛ^T| (escala) = {np.max(np.abs(LLt)):.3f}")

Si la diferencia entre $\hat\Lambda$ y $\Lambda$ se debe a la indeterminación por rotación, tiene que existir una matriz ortogonal $R$ con $\hat\Lambda R\approx\Lambda$. La buscamos resolviendo el **problema de Procrustes ortogonal**, $\min_{R^\top R=I}\|\hat\Lambda R-\Lambda\|_F$, con `scipy.linalg.orthogonal_procrustes`:

In [ ]:
from scipy.linalg import orthogonal_procrustes

R, _ = orthogonal_procrustes(Lambda_hat, Lambda)

print("R =\n", R.round(3))
print(f"R^T R = I: {np.allclose(R.T @ R, np.eye(k))}, det R = {np.linalg.det(R):.3f}")
print(f"máx. |Λ̂R - Λ| = {np.max(np.abs(Lambda_hat @ R - Lambda)):.3f}")

graficar_cargas([Lambda, Lambda_hat @ R], [r"$\Lambda$ (simulación)", r"$\hat\Lambda R$ (Procrustes)"])
plt.show()

**Para discutir:** ¿Coinciden las cargas estimadas con las de la simulación? ¿Y $\hat\Lambda\hat\Lambda^\top$?

#### b) Rotación varimax

La rotación **varimax** elige, entre todas las $\hat\Lambda R$ equivalentes, la que maximiza la varianza de los cuadrados de las cargas dentro de cada columna. Eso empuja cada carga hacia $0$ o hacia valores grandes, de modo que cada variable quede asociada principalmente a un factor.

Nuestra clase no rota las cargas: usamos `FactorAnalysis(rotation="varimax")` de `scikit-learn` y comparamos con nuestro ajuste.

In [ ]:
fa_v = FactorAnalysis(n_components=k, rotation="varimax", tol=1e-10, max_iter=10000).fit(X)
Lambda_v = fa_v.components_.T  # (p, k)

graficar_cargas(
    [Lambda, Lambda_hat, Lambda_v],
    [r"$\Lambda$ (simulación)", r"$\hat\Lambda$ (myFactorAnalysis)", r"$\hat\Lambda$ (varimax)"],
)
plt.show()

print(f"máx. |Λ̂_v Λ̂_v^T - Λ̂Λ̂^T| = {np.max(np.abs(Lambda_v @ Lambda_v.T - LLt_hat)):.1e}")
print(f"máx. |ψ̂_v - ψ̂| = {np.max(np.abs(fa_v.noise_variance_ - fa.psi)):.1e}")
print(f"Log-verosimilitud: myFactorAnalysis {fa.ell:.4f}, varimax {fa_v.score(X) * n:.4f}")

**Para discutir:** ¿Qué cambia con la rotación varimax y qué no?

#### c) Varianzas específicas

In [ ]:
comunalidad = np.sum(Lambda**2, axis=1)
comunalidad_hat = np.sum(Lambda_hat**2, axis=1)

pd.DataFrame(
    {
        "ψ_j": psi,
        "ψ̂_j": fa.psi,
        "comunalidad": comunalidad,
        "comunalidad estimada": comunalidad_hat,
        "varianza muestral": X.var(axis=0),
        "λ̂² + ψ̂": comunalidad_hat + fa.psi,
    },
    index=[f"x{j + 1}" for j in range(p)],
).round(3)

#### d) *(Adicional)* Análisis factorial frente a componentes principales

Repetimos la simulación con las mismas cargas, pero con una varianza específica mucho mayor en la variable 6: $\psi_6=9$, mientras que las demás quedan en $0.2$. La parte común de $x_6$ ($\lambda_6^2=0.49$) no cambia, pero ahora es una fracción chica de su varianza total.

In [ ]:
from sklearn.decomposition import PCA

psi_d = np.array([0.2, 0.2, 0.2, 0.2, 0.2, 9.0])
X_d, _ = simular_fa(n, mu, Lambda, psi_d, np.random.default_rng(1))

print("Varianzas muestrales:", X_d.var(axis=0).round(2))

# Sin estandarizar
fa_d = FactorAnalysis(n_components=2, rotation="varimax", random_state=0).fit(X_d)
pca_d = PCA(n_components=2).fit(X_d)

L_fa = fa_d.components_.T  # (p, 2) cargas
V_pca = pca_d.components_.T  # (p, 2) direcciones principales (columnas unitarias)

print(f"PCA: varianza explicada por cada componente = {pca_d.explained_variance_.round(2)} ({pca_d.explained_variance_ratio_.round(3)} del total)")
print("FA: ψ̂ =", fa_d.noise_variance_.round(2))

graficar_cargas([Lambda, L_fa, V_pca], [r"$\Lambda$ (simulación)", r"FA: $\hat\Lambda$ (varimax)", "PCA: direcciones"])
plt.show()

Las cargas de FA y las direcciones de PCA no están en la misma escala (las de PCA son vectores unitarios). Para comparar **direcciones**, miramos los subespacios que generan las columnas. Los **ángulos principales** entre dos subespacios de dimensión 2 (`scipy.linalg.subspace_angles`) son $0°$ si los subespacios coinciden y $90°$ si son ortogonales.

In [ ]:
from scipy.linalg import subspace_angles

print("Ángulos principales con span(Λ):")
print(f"  FA: {np.degrees(subspace_angles(L_fa, Lambda)).round(1)}°")
print(f"  PCA: {np.degrees(subspace_angles(V_pca, Lambda)).round(1)}°")

print("Primera dirección principal de PCA:", V_pca[:, 0].round(3))